# Minimal Character-Level RNN

In [47]:
import numpy as np

In [48]:
# training data
data = "hello world\n"
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print(f"data has {data_size} characters, {vocab_size} unique.")

data has 12 characters, 9 unique.


In [49]:
char_to_ix = {ch:i for i,ch in enumerate(chars)}
ix_to_char = {i:ch for i,ch in enumerate(chars)}

## Hyperparameters

In [50]:
hidden_size = 20
learning_rate = 0.1

## Model Parameters

In [51]:
Wxh = np.random.randn(hidden_size, vocab_size)*0.01
Whh = np.random.randn(hidden_size, hidden_size)*0.01
Why = np.random.randn(vocab_size, hidden_size)*0.01
bh = np.zeros((hidden_size,1))
by = np.zeros((vocab_size,1))
hprev = np.zeros((hidden_size,1))

In [52]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

## Training

In [53]:
for iter in range(2000):
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0

    # forward pass
    for t in range(len(data)-1):
        xs[t] = np.zeros((vocab_size,1))
        xs[t][char_to_ix[data[t]]] = 1

        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = softmax(ys[t])

        loss += -np.log(ps[t][char_to_ix[data[t+1]]])

    # backward pass
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)

    dhnext = np.zeros_like(hs[0])

    for t in reversed(range(len(data)-1)):

        dy = np.copy(ps[t])
        dy[char_to_ix[data[t+1]]] -= 1

        dWhy += np.dot(dy, hs[t].T)
        dby += dy

        dh = np.dot(Why.T, dy) + dhnext
        dhraw = (1 - hs[t]*hs[t]) * dh

        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)

        dhnext = np.dot(Whh.T, dhraw)

    # gradient clipping
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)

    # update weights
    Wxh -= learning_rate * dWxh
    Whh -= learning_rate * dWhh
    Why -= learning_rate * dWhy
    bh  -= learning_rate * dbh
    by  -= learning_rate * dby

    hprev = hs[len(data)-2]

    if iter % 200 == 0:
        print("loss:", loss)

loss: [24.1657788]
loss: [0.0551838]
loss: [0.02281826]
loss: [0.01416466]
loss: [0.01020435]
loss: [0.00794659]
loss: [0.00649237]
loss: [0.00547959]
loss: [0.0047348]
loss: [0.00416466]
